# Event-chain model — from the event repository to a trainable dataset

**What this notebook does, in order**

| Step | What happens | Uses the LLM? |
|---|---|---|
| 1 | Settings (the only cell you must edit) | – |
| 2 | Load the event repository | – |
| 3 | Connect to Ollama and check the model answers | yes (1 tiny call) |
| 4 | Base event → is it new or part of a trailing story? → **base cluster** | yes |
| 5 | Find the top similar past events | – |
| 6 | For each one: 15 days before (**event window**) and what came after, summarised into topic clusters | yes |
| 7 | Relevance score: base cluster vs every window cluster → saved as JSON | – |
| 8 | Counting predictor: which topics followed, weighted by relevance | – |
| 9 | Qwen reasons over the evidence and names the likely next events | yes |
| 10 | Repeat steps 4–7 over history (resumable) | yes, most of the run time |
| 11 | Build the training files (`train.jsonl`, `val.jsonl`, `test.jsonl`) | yes if rationales are on |
| 12 | Evaluate on the held-out period | optional |
| 13 | Fine-tune with LoRA | – (needs a GPU) |
| 14 | Pack the result for Ollama | – |

**Rules built into every step**
- Nothing dated on or after an event is used to describe what came *before* it.
- A past event is only used as an analogue once its own "what came after" period has fully passed.
- Every LLM result is cached on disk, one file per event. Re-running a cell continues where it stopped.
- The output is about **which kind of event tends to follow**. It is not a price forecast and not proof of cause.

**First run:** the notebook ships with `DRY_RUN = True`. Run everything once as it is. It uses no LLM
(clusters are made from the existing event labels) and finishes in about a minute. It proves the data loads
and every step connects. Its results mean nothing. Then set `DRY_RUN = False` in step 1, set `LLM_MODEL` to
your Qwen name, and run again.

## 1. Settings

In [ ]:
import os

# --- Data: one or more files or folders (.csv, .json, .jsonl). Several paths are joined into one repository.
# Oil layout example: ["memory_2001_2023/price_event_memory.json", "memory_2024_2026/df_major_memory_2024_2026.jsonl"]
DATA_PATHS = ["../scripts/causal_gold_articles_with_topics_bert.csv"]

# Column names tried for each field, first match wins. All matching text columns are joined.
FIELD_CANDIDATES = {
    "id":         ["event_id", "Doc_ID", "id"],
    "date":       ["date", "Date"],
    "event_type": ["Event type", "Event_type", "Event Types", "category", "event_type", "assigned_topic_bert"],
    "embedding":  ["bert_embeddings", "embedding", "gold_general_embedding"],
    "text":       ["text", "combined_text", "headline", "Headline", "gold_general_summary"],
}
# Rows whose text contains any of these are not real events and are left out. The second one matters:
# every such row carries the same embedding, so they would all look identical to each other.
DROP_TEXT_CONTAINING = ["Search Summary", "No gold-related content found"]
# Dates that are placeholders, not real event dates. In the gold file 71 rows carry the export date
# 2025-04-23 instead of their own. Set to [] for other data.
DROP_DATES = ["2025-04-23"]
# An embedding shared by more rows than this is the embedding of a placeholder text (for example an empty
# summary), not of an event. Those rows are left out.
MAX_SAME_EMBEDDING = 5

# --- Where results and caches are written
WORK_DIR = "chain_runs/run_v1"

# --- LLM (Ollama)
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
LLM_MODEL = "qwen3:14b"            # the exact name shown by `ollama list` on your machine
LLM_THINK = True                   # thinking mode for the reasoning steps; set None if your model has no thinking mode
LLM_NUM_CTX = 16384                # context window sent to Ollama
LLM_TIMEOUT_S = 600
LLM_RETRIES = 2
N_WORKERS = 4                      # parallel LLM calls (match OLLAMA_NUM_PARALLEL on the server)
DRY_RUN = True                     # True = no LLM and no embedding model, for a plumbing test. Set False for the real run.

# --- Embedding used for relevance scoring (summary vs summary)
# "sentence_transformers": the same logic as scripts/gold_relvant_data_extraction.ipynb (needs the model on disk)
# "ollama":                an embedding model served by Ollama, e.g. EMBED_MODEL = "all-minilm"
# "member_mean":           no model at all: average of the stored embeddings of the events in each cluster
EMBED_BACKEND = "sentence_transformers"
EMBED_MODEL = "all-MiniLM-L6-v2"   # model name, or a local folder path on a machine without internet

# --- Method
TOP_K = 50                 # similar past events per base event
WINDOW_DAYS_BEFORE = 15    # event window: days before each similar event
DAYS_AFTER = 15            # "what came next": days after each similar event
TRAIL_DAYS = 3             # how far back to look when asking "is the base event part of a running story?"
TRAIL_SIM_PCT = 97         # same story = more similar to the base event than this % of random event pairs
MAX_TRAIL = 8
MIN_GAP_DAYS = 3           # similar events closer than this to an already-picked one are skipped (same story twice)
SIM_FLOOR = 0.0            # minimum similarity for a past event to count as similar
WINDOW_SIM_PCT = 75        # an event belongs to a window only if it is more similar to the window's own event
                           # than this % of random event pairs (0 = take every event in the period;
                           # then almost every topic "follows" everything and there is nothing to learn)
MAX_EVENTS_PER_WINDOW = 30
TEXT_CHARS = 300           # characters of each event shown to the LLM
K_REASON = 10              # windows shown to the LLM in the reasoning prompt
REL_POWER = 1.0            # vote weight = relevance ** REL_POWER (step 12 tests other values)
TOPICS = None              # fixed topic list; None = take the event types found in the repository

# --- Which base event to walk through in steps 4–9 (None = the most recent event)
BASE_EVENT_ID = None

# --- History batch (steps 10–12)
MAX_ANCHORS = 150          # most recent N usable events; None = all of history (long run)
MIN_ANALOGUES = 10
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15
GENERATE_RATIONALES = True # the LLM writes the "reasoning" text of each training answer
N_EVAL_LLM = 30            # test events answered by the LLM in step 12 (0 = skip)

# --- Fine-tune (step 13)
RUN_FINETUNE = False
BASE_MODEL_HF = "/path/to/qwen-in-huggingface-format"   # NOT the Ollama file: the original weights folder
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
TRAIN_EPOCHS, LEARNING_RATE, MAX_TOKENS = 3, 2e-4, 6000

PROMPT_VERSION = "v1"      # change this if you edit a prompt, so old cached answers are not reused

## 2. Load the event repository

In [ ]:
import ast, hashlib, json, re, threading, time, urllib.error, urllib.request
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd

if not OLLAMA_HOST.startswith("http"):
    OLLAMA_HOST = "http://" + OLLAMA_HOST
OLLAMA_HOST = OLLAMA_HOST.rstrip("/")

WORK = Path(WORK_DIR)
_slug = lambda s: re.sub(r"[^A-Za-z0-9._-]", "_", str(s))
CACHE = WORK / "cache" / ("dryrun" if DRY_RUN else f"{_slug(LLM_MODEL)}_{PROMPT_VERSION}")
for sub in ("base_clusters", "windows", "rationales", "reasoning"):
    (CACHE / sub).mkdir(parents=True, exist_ok=True)
(WORK / "base_events").mkdir(parents=True, exist_ok=True)

try:
    show = display          # inside Jupyter
except NameError:
    show = lambda x: print(x.to_string() if hasattr(x, "to_string") else x)


def read_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None


def write_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + f".tmp{threading.get_ident()}")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=1), encoding="utf-8")
    os.replace(tmp, path)   # a half-written file never replaces a good one


def _read_one(path):
    path = Path(path)
    if path.is_dir():
        files = sorted(p for p in path.rglob("*") if p.suffix.lower() in (".json", ".jsonl", ".csv"))
        if not files:
            raise FileNotFoundError(f"No .json/.jsonl/.csv files inside {path}")
        return pd.concat([_read_one(p) for p in files], ignore_index=True)
    if not path.exists():
        raise FileNotFoundError(f"DATA_PATHS entry not found: {path.resolve()}")
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() == ".jsonl":
        return pd.DataFrame([json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()])
    obj = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(obj, dict):
        lists = [v for v in obj.values() if isinstance(v, list) and v and isinstance(v[0], dict)]
        obj = lists[0] if lists else [obj]
    return pd.DataFrame(obj)


def _parse_embedding(v):
    if isinstance(v, str):
        try:
            v = json.loads(v)
        except ValueError:
            try:
                v = ast.literal_eval(v)
            except (ValueError, SyntaxError):
                return None
    try:
        a = np.asarray(v, dtype=np.float32)
    except (TypeError, ValueError):
        return None
    return a if a.ndim == 1 and a.size > 1 and np.isfinite(a).all() else None


def _parse_dates(col):
    try:
        return pd.to_datetime(col, errors="coerce", format="mixed")
    except (TypeError, ValueError):
        return pd.to_datetime(col, errors="coerce")


raw = pd.concat([_read_one(p) for p in DATA_PATHS], ignore_index=True)
cols = {}
for field, candidates in FIELD_CANDIDATES.items():
    found = [c for c in candidates if c in raw.columns]
    if not found:
        raise KeyError(f"No column for '{field}'. Tried {candidates}. Columns in your data: {list(raw.columns)}. "
                       f"Add the right name to FIELD_CANDIDATES['{field}'].")
    cols[field] = found if field == "text" else found[0]
print("Columns used:", cols)


def _join_text(row):
    """All text columns as one line: repeated lines, the event's own id and 'UNKNOWN' placeholders removed."""
    own_id = str(row[cols["id"]]).strip().lower()
    lines = []
    for c in cols["text"]:
        if pd.notna(row[c]):
            lines += [l.strip() for l in str(row[c]).splitlines()]
    lines = [l for l in lines if l and l.lower() not in (own_id, "unknown", "nan")]
    return " | ".join(dict.fromkeys(lines))


ev = pd.DataFrame({
    "id": raw[cols["id"]].astype(str),
    "date": _parse_dates(raw[cols["date"]]),
    "event_type": raw[cols["event_type"]].astype(str).str.strip(),
    "text": raw.apply(_join_text, axis=1),
    "embedding": raw[cols["embedding"]].map(_parse_embedding),
})
n_raw = len(ev)
ev = ev[ev["date"].notna() & ev["embedding"].notna() & (ev["text"].str.len() > 0)]
for pat in DROP_TEXT_CONTAINING:
    ev = ev[~ev["text"].str.contains(pat, case=False, regex=False)]
for d in DROP_DATES:
    placeholder = ev["date"].dt.strftime("%Y-%m-%d") == d
    print(f"Left out {int(placeholder.sum())} rows dated {d} (listed in DROP_DATES)")
    ev = ev[~placeholder]
dims = ev["embedding"].map(len)
ev = ev[dims == dims.mode().iloc[0]]
emb_key = ev["embedding"].map(lambda a: hashlib.md5(a.tobytes()).hexdigest())
shared = emb_key.map(emb_key.value_counts()) > MAX_SAME_EMBEDDING
print(f"Left out {int(shared.sum())} rows whose embedding is shared by more than {MAX_SAME_EMBEDDING} rows (placeholder text)")
ev = ev[~shared]
ev = ev.drop_duplicates("id").sort_values(["date", "id"], kind="stable").reset_index(drop=True)
if len(ev) < 100:
    raise ValueError(f"Only {len(ev)} usable events out of {n_raw}. Check dates, embeddings and text columns.")

ids = ev["id"].tolist()
texts = ev["text"].tolist()
types = ev["event_type"].tolist()
date_str = ev["date"].dt.strftime("%Y-%m-%d").tolist()
day = ev["date"].values.astype("datetime64[D]").astype(np.int64)      # days since 1970, sorted ascending
E = np.stack(ev["embedding"].values).astype(np.float32)
E /= np.linalg.norm(E, axis=1, keepdims=True) + 1e-12
idx_of = {e: i for i, e in enumerate(ids)}
LAST_DAY = int(day[-1])

if TOPICS is None:
    TOPICS = sorted(t for t in set(types) if t and t.lower() not in ("nan", "none"))
TOPIC_ENUM = list(TOPICS) + ([] if "Other" in TOPICS else ["Other"])

_rng = np.random.default_rng(0)
_a, _b = _rng.integers(0, len(E), 50000), _rng.integers(0, len(E), 50000)
_pair = np.einsum("ij,ij->i", E[_a], E[_b])[_a != _b]
TRAIL_SIM = float(np.percentile(_pair, TRAIL_SIM_PCT))
WINDOW_MIN_SIM = float(np.percentile(_pair, WINDOW_SIM_PCT)) if WINDOW_SIM_PCT > 0 else -1.0

print(f"{len(ev):,} usable events of {n_raw:,} rows · {date_str[0]} to {date_str[-1]} · embedding size {E.shape[1]}")
print(f"{len(TOPICS)} topics:", TOPICS if len(TOPICS) <= 25 else TOPICS[:25] + ["..."])
if len(TOPICS) > 25:
    print("More than 25 topics makes counting thin. Consider setting TOPICS in step 1 to a shorter fixed list.")
print(f"Similarity between two random events: median {np.median(_pair):.2f}. "
      f"Same-story threshold {TRAIL_SIM:.2f} · window threshold {WINDOW_MIN_SIM:.2f}")

## 3. Connect to the LLM and the embedding model

In [ ]:
_think_ok = True
_fail_lock = threading.Lock()
_fail_streak = 0


def _post(route, body, timeout=None):
    req = urllib.request.Request(OLLAMA_HOST + route, data=json.dumps(body).encode("utf-8"),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout or LLM_TIMEOUT_S) as r:
        return json.loads(r.read().decode("utf-8"))


def ollama_chat(messages, schema=None, think=None):
    """One chat call. `schema` forces the answer to be JSON of that shape."""
    global _think_ok
    body = {"model": LLM_MODEL, "messages": messages, "stream": False,
            "options": {"temperature": 0, "num_ctx": LLM_NUM_CTX}}
    if schema is not None:
        body["format"] = schema
    if think is not None and _think_ok:
        body["think"] = think
    try:
        return _post("/api/chat", body)["message"]
    except urllib.error.HTTPError as e:
        detail = e.read().decode("utf-8", "replace")
        if e.code == 400 and "think" in body and "think" in detail.lower():
            _think_ok = False
            print(f"Note: {LLM_MODEL} does not accept the thinking switch; continuing without it.")
            return ollama_chat(messages, schema, None)
        raise RuntimeError(f"Ollama returned {e.code}: {detail[:300]}") from e


def llm_json(system, user, schema, validate, fake, think=False, tag=""):
    """Returns the parsed JSON answer, or None after retries (the failure is written to failures.jsonl)."""
    global _fail_streak
    if DRY_RUN:
        return fake()
    problem = None
    for _ in range(LLM_RETRIES + 1):
        try:
            msg = ollama_chat([{"role": "system", "content": system}, {"role": "user", "content": user}], schema, think)
            content = re.sub(r"<think>.*?</think>", "", msg.get("content", ""), flags=re.S).strip()
            out = json.loads(content)
            problem = validate(out)
            if problem is None:
                with _fail_lock:
                    _fail_streak = 0
                return out
        except Exception as e:
            problem = f"{type(e).__name__}: {e}"
    with _fail_lock:
        _fail_streak += 1
        with open(WORK / "failures.jsonl", "a", encoding="utf-8") as f:
            f.write(json.dumps({"tag": tag, "problem": str(problem)[:500], "time": time.strftime("%F %T")}) + "\n")
        if _fail_streak >= 10:
            raise RuntimeError(f"10 LLM calls failed in a row. Last problem: {problem}. "
                               f"Check Ollama at {OLLAMA_HOST} and the model name '{LLM_MODEL}'.")
    return None


_st_model = None
_vec_cache = {}


def embed_texts(text_list):
    global _st_model
    need = [t for t in dict.fromkeys(text_list) if t not in _vec_cache]
    for start in range(0, len(need), 64):
        chunk = need[start:start + 64]
        if EMBED_BACKEND == "sentence_transformers":
            if _st_model is None:
                from sentence_transformers import SentenceTransformer
                _st_model = SentenceTransformer(EMBED_MODEL)
            vecs = _st_model.encode(chunk, convert_to_numpy=True, show_progress_bar=False)
        elif EMBED_BACKEND == "ollama":
            vecs = np.asarray(_post("/api/embed", {"model": EMBED_MODEL, "input": chunk})["embeddings"])
        else:
            raise ValueError(f"EMBED_BACKEND '{EMBED_BACKEND}' cannot embed text")
        for t, v in zip(chunk, np.asarray(vecs, dtype=np.float32)):
            _vec_cache[t] = v / (np.linalg.norm(v) + 1e-12)
    return np.stack([_vec_cache[t] for t in text_list])


USE_MEMBER_MEAN = DRY_RUN or EMBED_BACKEND == "member_mean"

if DRY_RUN:
    print("DRY RUN: no LLM and no embedding model. Clusters come from the existing event labels; "
          "relevance uses the stored event embeddings. Results are for testing the plumbing only.")
else:
    try:
        with urllib.request.urlopen(OLLAMA_HOST + "/api/tags", timeout=30) as r:
            available = [m["name"] for m in json.loads(r.read().decode("utf-8")).get("models", [])]
    except Exception as e:
        raise RuntimeError(f"Cannot reach Ollama at {OLLAMA_HOST} ({e}). Start `ollama serve` on the node, "
                           f"or set OLLAMA_HOST in step 1.") from e
    if LLM_MODEL not in available:
        raise RuntimeError(f"Model '{LLM_MODEL}' is not in Ollama. Set LLM_MODEL to one of: {available}")
    check = llm_json("Reply in JSON.", "Is this a connection test? Answer true.",
                     {"type": "object", "properties": {"ok": {"type": "boolean"}}, "required": ["ok"]},
                     lambda o: None if isinstance(o.get("ok"), bool) else "no 'ok' field", fake=None, tag="preflight")
    if check is None:
        raise RuntimeError(f"{LLM_MODEL} answered but not in the requested JSON shape. See {WORK / 'failures.jsonl'}.")
    print(f"LLM ready: {LLM_MODEL} at {OLLAMA_HOST}")
    if not USE_MEMBER_MEAN:
        try:
            dim = embed_texts(["connection test"]).shape[1]
        except Exception as e:
            raise RuntimeError(f"Embedding backend '{EMBED_BACKEND}' with model '{EMBED_MODEL}' failed: {e}. "
                               f"Options: point EMBED_MODEL at a local model folder, use EMBED_BACKEND='ollama' "
                               f"with a pulled embedding model, or EMBED_BACKEND='member_mean' (needs no model).") from e
        print(f"Embedding ready: {EMBED_BACKEND} / {EMBED_MODEL} ({dim} numbers per text)")

## 4. Base event → base cluster

The base event is compared with the events of the previous `TRAIL_DAYS` days. Any that are similar enough
(`TRAIL_SIM_PCT`) are treated as the same running story, and the LLM summarises them together. If there are none,
the base event is new and is summarised on its own. Events on the same day are left out, because the data has
dates but no times, so their order is unknown.

In [ ]:
SYS_SUMMARY = ("You organise commodity-market news events. Use only the events given. Do not add facts, "
               "prices or outcomes that are not in the text. Answer in JSON.")
TOPIC_LINE = "Allowed topics: " + "; ".join(TOPIC_ENUM) + "."

BASE_SCHEMA = {"type": "object", "properties": {
    "topic": {"type": "string", "enum": TOPIC_ENUM}, "summary": {"type": "string"}}, "required": ["topic", "summary"]}
GROUP_SCHEMA = {"type": "object", "properties": {"clusters": {"type": "array", "items": {
    "type": "object", "properties": {
        "topic": {"type": "string", "enum": TOPIC_ENUM}, "summary": {"type": "string"},
        "event_numbers": {"type": "array", "items": {"type": "integer"}}},
    "required": ["topic", "summary", "event_numbers"]}}}, "required": ["clusters"]}


def event_lines(indices):
    return "\n".join(f"[{n}] {date_str[i]} | label: {types[i]} | {texts[i][:TEXT_CHARS]}" for n, i in enumerate(indices, 1))


def topic_or_other(t):
    return t if t in TOPIC_ENUM else "Other"


def trailing_members(i):
    lo = np.searchsorted(day, day[i] - TRAIL_DAYS, side="left")
    hi = np.searchsorted(day, day[i], side="left")            # strictly before the base event's day
    if lo >= hi:
        return []
    sims = E[lo:hi] @ E[i]
    keep = [(float(s), lo + k) for k, s in enumerate(sims) if s >= TRAIL_SIM]
    return sorted(j for _, j in sorted(keep, reverse=True)[:MAX_TRAIL])


def base_cluster(i):
    path = CACHE / "base_clusters" / f"{_slug(ids[i])}.json"
    hit = read_json(path)
    if hit:
        return hit
    members = trailing_members(i)
    user = (f"{TOPIC_LINE}\n\nBASE EVENT:\n{event_lines([i])}\n\n"
            + (f"EARLIER EVENTS FROM THE SAME STORY:\n{event_lines(members)}\n\n" if members else "")
            + "Write one summary (2–3 sentences) of the situation as of the base event, and choose its topic.")
    out = llm_json(SYS_SUMMARY, user, BASE_SCHEMA,
                   lambda o: None if o.get("topic") in TOPIC_ENUM and str(o.get("summary", "")).strip() else "bad topic or empty summary",
                   fake=lambda: {"topic": topic_or_other(types[i]), "summary": texts[i][:240]},
                   tag=f"base:{ids[i]}")
    if out is None:
        return None
    rec = {"event_id": ids[i], "date": date_str[i], "status": "continuation" if members else "new",
           "trailing_event_ids": [ids[j] for j in members], "topic": out["topic"], "summary": out["summary"].strip()}
    write_json(path, rec)
    return rec


if BASE_EVENT_ID and BASE_EVENT_ID not in idx_of:
    raise KeyError(f"BASE_EVENT_ID '{BASE_EVENT_ID}' is not in the repository. Example ids: {ids[-3:]}")
BASE = idx_of[BASE_EVENT_ID] if BASE_EVENT_ID else len(ids) - 1
print(f"Base event: {ids[BASE]} · {date_str[BASE]} · {types[BASE]}\n  {texts[BASE][:200]}\n")
bc = base_cluster(BASE)
if bc is None:
    raise RuntimeError(f"The LLM could not summarise the base event. See {WORK / 'failures.jsonl'}.")
print(f"Status: {bc['status']} ({len(bc['trailing_event_ids'])} trailing events)\nTopic: {bc['topic']}\nSummary: {bc['summary']}")

## 5. Find the similar past events

Same search as before: cosine similarity on the stored event embeddings. Two extra rules:
- a past event only qualifies if its own "what came next" period (`DAYS_AFTER`) ended before the base event;
- events within `MIN_GAP_DAYS` of an already-picked one are skipped, so one story is not counted several times.

In [ ]:
def retrieve(i, top_k=TOP_K):
    n_ok = int(np.searchsorted(day, day[i] - DAYS_AFTER - 1, side="right"))
    if n_ok == 0:
        return []
    sims = E[:n_ok] @ E[i]
    picked, picked_days = [], []
    for j in np.argsort(-sims):
        if sims[j] < SIM_FLOOR or len(picked) == top_k:
            break
        if MIN_GAP_DAYS and any(abs(int(day[j]) - d) < MIN_GAP_DAYS for d in picked_days):
            continue
        picked.append((int(j), float(sims[j])))
        picked_days.append(int(day[j]))
    return picked


analogues = retrieve(BASE)
if not analogues:
    raise RuntimeError(f"No past events qualify for {ids[BASE]} ({date_str[BASE]}). Choose a later BASE_EVENT_ID.")
print(f"{len(analogues)} similar past events for {ids[BASE]}")
show(pd.DataFrame([{"event_id": ids[j], "date": date_str[j], "similarity": round(s, 4), "label": types[j],
                    "text": texts[j][:90]} for j, s in analogues[:10]]))

## 6. Event windows: what came before and after each similar event

For every similar event the LLM gets two lists: the related events of the 15 days before it, and the related
events of the 15 days after it ("related" is set by `WINDOW_SIM_PCT`). It groups each list into topic clusters and writes a short summary per cluster.
One file per event is saved under `cache/.../windows/`, so every event is summarised once, whichever base
event later needs it.

In [ ]:
WINDOW_RULE = f"before{WINDOW_DAYS_BEFORE}-after{DAYS_AFTER}-simpct{WINDOW_SIM_PCT}-max{MAX_EVENTS_PER_WINDOW}"


def window_members(i, start_off, end_off):
    lo = np.searchsorted(day, day[i] + start_off, side="left")
    hi = np.searchsorted(day, day[i] + end_off, side="right")
    members = [j for j in range(lo, hi) if j != i]
    if not members:
        return []
    sims = E[members] @ E[i]
    ranked = [members[k] for k in np.argsort(-sims) if sims[k] >= WINDOW_MIN_SIM]
    return sorted(ranked[:MAX_EVENTS_PER_WINDOW])


def _validate_groups(n):
    def check(o):
        cl = o.get("clusters")
        if not isinstance(cl, list) or not cl:
            return "no clusters"
        for c in cl:
            if c.get("topic") not in TOPIC_ENUM or not str(c.get("summary", "")).strip():
                return "bad topic or empty summary"
        if not any(isinstance(k, int) and 1 <= k <= n for c in cl for k in c.get("event_numbers", [])):
            return "no valid event numbers"
        return None
    return check


def _fake_groups(members):
    by_topic = {}
    for n, j in enumerate(members, 1):
        by_topic.setdefault(topic_or_other(types[j]), []).append(n)
    return {"clusters": [{"topic": t, "summary": texts[members[nums[0] - 1]][:200], "event_numbers": nums}
                         for t, nums in by_topic.items()]}


def summarise_group(anchor, members, side):
    """Clusters for one side of a window. [] = no events there. None = the LLM failed."""
    if not members:
        return []
    user = (f"{TOPIC_LINE}\n\nEVENTS ({side} the reference date {date_str[anchor]}):\n{event_lines(members)}\n\n"
            "Group these events into clusters that each describe one development. For each cluster give its topic, "
            "a 1–2 sentence summary, and the numbers of the events in it. Leave out events that are not market-relevant.")
    out = llm_json(SYS_SUMMARY, user, GROUP_SCHEMA, _validate_groups(len(members)),
                   fake=lambda: _fake_groups(members), tag=f"window-{side}:{ids[anchor]}")
    if out is None:
        return None
    clusters = []
    for c in out["clusters"]:
        nums = sorted({k for k in c.get("event_numbers", []) if isinstance(k, int) and 1 <= k <= len(members)})
        if not nums:
            continue
        idx = [members[k - 1] for k in nums]
        offs = [int(day[j] - day[anchor]) for j in idx]
        clusters.append({"topic": c["topic"], "summary": c["summary"].strip(), "event_ids": [ids[j] for j in idx],
                         "first_offset_days": min(offs), "last_offset_days": max(offs)})
    return clusters


def get_window(i):
    path = CACHE / "windows" / f"{_slug(ids[i])}.json"
    after_complete = int(day[i]) + DAYS_AFTER <= LAST_DAY
    hit = read_json(path)
    if hit and hit.get("rule") == WINDOW_RULE and (hit["after_complete"] or not after_complete):
        return hit
    before = summarise_group(i, window_members(i, -WINDOW_DAYS_BEFORE, -1), "before")
    after = summarise_group(i, window_members(i, 1, DAYS_AFTER), "after") if after_complete else []
    if before is None or after is None:
        return None
    rec = {"event_id": ids[i], "date": date_str[i], "rule": WINDOW_RULE,
           "after_complete": after_complete, "before": before, "after": after}
    write_json(path, rec)
    return rec


def run_parallel(fn, items, label):
    items = list(items)
    if not items:
        return
    t0, done = time.time(), 0
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        for _ in ex.map(fn, items):
            done += 1
            if done % max(50, len(items) // 20) == 0 or done == len(items):
                rate = done / max(time.time() - t0, 1e-9)
                print(f"  {label}: {done}/{len(items)} · {rate:.2f}/s · about {(len(items) - done) / rate / 60:.0f} min left")


def cached_window(i):
    """The saved window of event i, or None if it is missing or was built with different window settings."""
    hit = read_json(CACHE / "windows" / f"{_slug(ids[i])}.json")
    return hit if hit and hit.get("rule") == WINDOW_RULE else None


def missing_windows(indices):
    return [i for i in indices if cached_window(i) is None]


todo = missing_windows([j for j, _ in analogues])
print(f"{len(todo)} windows to summarise (2 LLM calls each); {len(analogues) - len(todo)} already cached")
run_parallel(get_window, todo, "windows")
w0 = get_window(analogues[0][0])
if w0 is None:
    raise RuntimeError(f"The LLM could not summarise the first window. See {WORK / 'failures.jsonl'}.")
print(f"\nExample — most similar past event {w0['event_id']} ({w0['date']})")
for side in ("before", "after"):
    print(f" {side.upper()}:")
    for c in w0[side]:
        print(f"   [{c['topic']}] day {c['first_offset_days']:+d}: {c['summary'][:150]}")

## 7. Relevance: base cluster vs every window cluster

Same scoring as `summarize_and_score` in `scripts/gold_relvant_data_extraction.ipynb`: embed both summaries,
take the cosine, and map it to 0.1–1.0. The difference is the reference text: there it was a fixed gold
sentence, here it is the base cluster summary. A window's relevance is that of its best-matching cluster.
The full result for the base event is saved as one JSON file.

In [ ]:
def normalize_similarity(cos_sim):
    return round(0.1 + 0.9 * (float(cos_sim) + 1) / 2, 4)


def _mean_vec(event_ids):
    v = E[[idx_of[e] for e in event_ids]].mean(axis=0)
    return v / (np.linalg.norm(v) + 1e-12)


def analyse(i):
    """Base cluster + similar events + their windows + relevance. Uses cached summaries only."""
    b = base_cluster(i)
    if b is None:
        return None
    found = retrieve(i)
    wins = [(j, s, cached_window(j)) for j, s in found]
    wins = [(j, s, w) for j, s, w in wins if w is not None]
    if USE_MEMBER_MEAN:
        base_vec = _mean_vec([ids[i]] + b["trailing_event_ids"])
        vec = lambda c: _mean_vec(c["event_ids"])
    else:
        embed_texts([b["summary"]] + [c["summary"] for _, _, w in wins for c in w["before"]])
        base_vec = _vec_cache[b["summary"]]
        vec = lambda c: _vec_cache[c["summary"]]
    rows = []
    for j, s, w in wins:
        scored = sorted(({**c, "relevance": normalize_similarity(base_vec @ vec(c))} for c in w["before"]),
                        key=lambda c: -c["relevance"])
        rows.append({"event_id": ids[j], "date": date_str[j], "label": types[j], "text": texts[j][:TEXT_CHARS],
                     "similarity": round(s, 4), "empty_window": not scored,
                     "window_relevance": scored[0]["relevance"] if scored else 0.1,
                     "before_clusters": scored, "after_clusters": w["after"]})
    return {"base": b, "n_retrieved": len(found), "n_with_window": len(rows), "analogues": rows}


res = analyse(BASE)
print(f"{res['n_with_window']} of {res['n_retrieved']} similar events have a summarised window")
show(pd.DataFrame([{"event_id": r["event_id"], "date": r["date"], "similarity": r["similarity"],
                    "window_relevance": r["window_relevance"],
                    "best_matching_cluster": (r["before_clusters"][0]["topic"] + ": " + r["before_clusters"][0]["summary"][:70]) if r["before_clusters"] else "(nothing in the 15 days before)",
                    "then_followed": ", ".join(sorted({c["topic"] for c in r["after_clusters"]})) or "(nothing)"}
                   for r in sorted(res["analogues"], key=lambda r: -r["window_relevance"])[:12]]))

## 8. Counting predictor: what followed, weighted by relevance

The event-chain twin of the KDE. The KDE weighs the *returns* that followed the similar events;
this weighs the *topics* that followed them. `share` is the weighted share of similar events that were followed
by that topic within `DAYS_AFTER` days. Several topics can follow one event, so shares do not add up to 100%.

In [ ]:
def vote(rows, power=None):
    power = REL_POWER if power is None else power
    score, first_days, total = Counter(), {}, 0.0
    for r in rows:
        w = r["window_relevance"] ** power
        total += w
        seen = {}
        for c in r["after_clusters"]:
            seen[c["topic"]] = min(seen.get(c["topic"], 10**6), c["first_offset_days"])
        for t, d in seen.items():
            score[t] += w
            first_days.setdefault(t, []).append(d)
    table = [{"topic": t, "share": round(s / total, 4), "n_events": len(first_days[t]),
              "typical_days_after": int(np.median(first_days[t]))} for t, s in score.items()] if total else []
    return sorted(table, key=lambda x: (-x["share"], x["topic"]))


table = vote(res["analogues"])
show(pd.DataFrame(table))

## 9. The LLM reasons over the evidence

The LLM sees the base cluster, the counting table, and the `K_REASON` most relevant windows with what followed
each one. It must answer from that evidence only. The same prompt shape is used later for the training files,
so what the model is trained on is exactly what it will be asked when deployed.

In [ ]:
SYS_REASON = ("You are a commodity-market analyst. You are given a current situation and past situations that "
              "resembled it, with what followed each one. Say which kinds of events are likely to follow now. "
              "Use only the evidence given. Refer to past cases by their date. Do not predict prices. "
              "If the evidence is thin or mixed, say so. Answer in JSON.")
REASON_SCHEMA = {"type": "object", "properties": {
    "reasoning": {"type": "string"},
    "likely_next": {"type": "array", "items": {"type": "object", "properties": {
        "topic": {"type": "string", "enum": TOPIC_ENUM}, "expected_within_days": {"type": "integer"},
        "description": {"type": "string"}}, "required": ["topic", "expected_within_days", "description"]}}},
    "required": ["reasoning", "likely_next"]}


def reason_prompt(res_i, table_i):
    b = res_i["base"]
    top = sorted(res_i["analogues"], key=lambda r: -r["window_relevance"])[:K_REASON]
    lines = [TOPIC_LINE, "", f"CURRENT SITUATION ({b['date']}, {b['status']} story, topic {b['topic']}):", b["summary"], "",
             f"WHAT FOLLOWED {res_i['n_with_window']} SIMILAR PAST SITUATIONS (share weighted by relevance):"]
    lines += [f"- {t['topic']}: {t['share']:.0%} of cases, typically {t['typical_days_after']} days later" for t in table_i[:8]] or ["- no follow-up events recorded"]
    lines += ["", f"THE {len(top)} MOST RELEVANT PAST SITUATIONS:"]
    for r in top:
        before = r["before_clusters"][0]["summary"] if r["before_clusters"] else "no events in the 15 days before"
        after = "; ".join(f"{c['topic']} (day +{c['first_offset_days']}): {c['summary']}" for c in r["after_clusters"]) or "nothing recorded"
        lines += [f"* {r['date']} · relevance {r['window_relevance']:.2f}", f"  event: {r['text'][:200]}",
                  f"  before: {before}", f"  then: {after}"]
    lines += ["", "Give your reasoning first (3–6 sentences), then the likely next events, most likely first (at most 4)."]
    return "\n".join(lines)


def _validate_reason(o):
    ln = o.get("likely_next")
    if not isinstance(ln, list) or not ln or not str(o.get("reasoning", "")).strip():
        return "missing reasoning or likely_next"
    return None if all(x.get("topic") in TOPIC_ENUM for x in ln) else "topic outside the allowed list"


def reason(i, res_i, table_i):
    path = CACHE / "reasoning" / f"{_slug(ids[i])}.json"
    hit = read_json(path)
    if hit:
        return hit
    fake = lambda: {"reasoning": "DRY RUN: this text is a placeholder, not model output.",
                    "likely_next": [{"topic": t["topic"], "expected_within_days": t["typical_days_after"],
                                     "description": "placeholder"} for t in table_i[:3]] or
                                   [{"topic": "Other", "expected_within_days": DAYS_AFTER, "description": "placeholder"}]}
    out = llm_json(SYS_REASON, reason_prompt(res_i, table_i), REASON_SCHEMA, _validate_reason, fake,
                   think=LLM_THINK, tag=f"reason:{ids[i]}")
    if out is not None:
        write_json(path, out)
    return out


answer = reason(BASE, res, table)
if answer is None:
    raise RuntimeError(f"The LLM did not return a usable answer. See {WORK / 'failures.jsonl'}.")
print("REASONING\n", answer["reasoning"], "\n\nLIKELY NEXT")
for x in answer["likely_next"]:
    print(f" - {x['topic']} within {x['expected_within_days']} days: {x['description']}")

out_file = WORK / "base_events" / f"{_slug(ids[BASE])}.json"
write_json(out_file, {**res, "counting_table": table, "llm_answer": answer,
                      "settings": {"top_k": TOP_K, "days_before": WINDOW_DAYS_BEFORE, "days_after": DAYS_AFTER,
                                   "llm": "dry-run" if DRY_RUN else LLM_MODEL, "embedding": "member_mean" if USE_MEMBER_MEAN else EMBED_MODEL}})
print(f"\nEverything for this base event is saved in {out_file}")

## 10. Run over history (the long step)

To learn, the model needs many past events treated as if each were the base event, with the true answer
next to it: what really followed. This cell summarises the windows and base clusters those examples need.
It can be stopped and re-run at any time. Start with a small `MAX_ANCHORS`; raise it (or set `None`) once
the pilot looks right.

In [ ]:
usable = [i for i in range(len(ids)) if int(day[i]) + DAYS_AFTER <= LAST_DAY]
retrieved = {}
anchors = []
for i in reversed(usable):                     # most recent first
    found = retrieve(i)
    if len(found) >= MIN_ANALOGUES:
        retrieved[i] = found
        anchors.append(i)
        if MAX_ANCHORS and len(anchors) == MAX_ANCHORS:
            break
anchors.sort()
needed = sorted(set(anchors) | {j for i in anchors for j, _ in retrieved[i]})
todo_w = missing_windows(needed)
todo_b = [i for i in anchors if not (CACHE / "base_clusters" / f"{_slug(ids[i])}.json").exists()]
print(f"{len(anchors)} anchor events ({date_str[anchors[0]]} to {date_str[anchors[-1]]}) need {len(needed)} windows.")
print(f"To do now: {len(todo_w)} windows (2 LLM calls each) and {len(todo_b)} base clusters (1 call each) "
      f"= about {2 * len(todo_w) + len(todo_b)} LLM calls.")
run_parallel(get_window, todo_w, "windows")
run_parallel(base_cluster, todo_b, "base clusters")
still = len(missing_windows(needed))
print(f"Done. {still} windows could not be summarised" + (f" — see {WORK / 'failures.jsonl'}" if still else "") + ".")

## 11. Build the training files

One example per anchor event:
- **question** = the reasoning prompt of step 9, built only from what was known before that event;
- **answer** = what really followed it (topics, days, summaries).

The split is by time: oldest events train, the next block validates, the newest block is the test.
Events whose "what came next" period crosses into the following block are dropped, so no answer leaks across.

With `GENERATE_RATIONALES = True` the LLM also writes the `reasoning` text of each answer. It is shown the
true outcome and asked to explain it from the evidence. Treat that text as a written justification:
the outcome is real, the explanation is the LLM's.

In [ ]:
SYS_TEACHER = ("You write the reasoning an analyst would have given BEFORE the outcome was known. You are shown the "
               "evidence the analyst had, and privately the events that really followed. Write 3–6 sentences that lead "
               "from the evidence to those events. Refer to past cases by their date. Never say or imply that you know "
               "the outcome. Do not mention prices. Answer in JSON.")
TEACHER_SCHEMA = {"type": "object", "properties": {"reasoning": {"type": "string"}}, "required": ["reasoning"]}

analysed = {}
for i in anchors:
    r = analyse(i)
    if r is None or r["n_with_window"] < MIN_ANALOGUES:
        continue
    own = cached_window(i)
    if own is None or not own["after_complete"] or not own["after"]:
        continue
    analysed[i] = {"res": r, "table": vote(r["analogues"]), "actual": sorted(own["after"], key=lambda c: c["first_offset_days"])}
kept = sorted(analysed)
print(f"{len(kept)} of {len(anchors)} anchors have enough evidence and at least one follow-up event")

n = len(kept)
if n < 10:
    raise RuntimeError(f"Only {n} usable anchors. Raise MAX_ANCHORS in step 1 (None = all of history).")
val_start, test_start = day[kept[int(n * TRAIN_FRAC)]], day[kept[int(n * (TRAIN_FRAC + VAL_FRAC))]]
split = {"train": [i for i in kept if day[i] + DAYS_AFTER < val_start],
         "val":   [i for i in kept if val_start <= day[i] and day[i] + DAYS_AFTER < test_start],
         "test":  [i for i in kept if day[i] >= test_start]}
for name, members in split.items():
    span = f"{date_str[members[0]]} to {date_str[members[-1]]}" if members else "—"
    print(f"  {name:5s}: {len(members):5d} events · {span}")


def rationale(i):
    path = CACHE / "rationales" / f"{_slug(ids[i])}.json"
    hit = read_json(path)
    if hit:
        return hit["reasoning"]
    a = analysed[i]
    truth = "\n".join(f"- {c['topic']} (day +{c['first_offset_days']}): {c['summary']}" for c in a["actual"])
    out = llm_json(SYS_TEACHER, reason_prompt(a["res"], a["table"]) + "\n\nWHAT REALLY FOLLOWED (private):\n" + truth,
                   TEACHER_SCHEMA, lambda o: None if str(o.get("reasoning", "")).strip() else "empty reasoning",
                   fake=lambda: {"reasoning": "DRY RUN: placeholder rationale."}, think=LLM_THINK, tag=f"rationale:{ids[i]}")
    if out is None:
        return None
    write_json(path, out)
    return out["reasoning"]


if GENERATE_RATIONALES:
    todo_r = [i for i in split["train"] + split["val"] if not (CACHE / "rationales" / f"{_slug(ids[i])}.json").exists()]
    print(f"Writing {len(todo_r)} rationales (1 LLM call each)")
    run_parallel(rationale, todo_r, "rationales")


def training_example(i, with_rationale):
    a = analysed[i]
    text = rationale(i) if with_rationale else ""
    if text is None:
        return None
    target = {"reasoning": text,
              "likely_next": [{"topic": c["topic"], "expected_within_days": c["first_offset_days"], "description": c["summary"]}
                              for c in a["actual"]]}
    return {"event_id": ids[i], "date": date_str[i],
            "messages": [{"role": "system", "content": SYS_REASON},
                         {"role": "user", "content": reason_prompt(a["res"], a["table"])},
                         {"role": "assistant", "content": json.dumps(target, ensure_ascii=False)}]}


DATASET = WORK / "dataset"
DATASET.mkdir(exist_ok=True)
for name, members in split.items():
    examples = [training_example(i, GENERATE_RATIONALES and name != "test") for i in members]
    examples = [e for e in examples if e is not None]
    with open(DATASET / f"{name}.jsonl", "w", encoding="utf-8") as f:
        for e in examples:
            f.write(json.dumps(e, ensure_ascii=False) + "\n")
    print(f"  wrote {len(examples)} examples to {DATASET / (name + '.jsonl')}")
write_json(DATASET / "split_manifest.json", {
    "val_starts": str(np.datetime64(int(val_start), "D")), "test_starts": str(np.datetime64(int(test_start), "D")),
    "counts": {k: len(v) for k, v in split.items()}, "topics": TOPIC_ENUM, "llm": "dry-run" if DRY_RUN else LLM_MODEL,
    "settings": {"top_k": TOP_K, "days_before": WINDOW_DAYS_BEFORE, "days_after": DAYS_AFTER, "k_reason": K_REASON}})

## 12. Evaluate on the held-out period

The question for every method: of the topics it ranked highest, did at least one really follow?
`hit@1` uses its first choice, `hit@3` its first three.

- **most common topics** — always answers with the overall most frequent follow-up topics. The floor to beat.
- **counting, unweighted** — the similar events' follow-ups, every event counted equally.
- **counting, relevance-weighted** — step 8. The weight power is chosen on the validation block, then fixed.
- **LLM with retrieval** — step 9, on the first `N_EVAL_LLM` test events.

After fine-tuning, point `LLM_MODEL` at the new model and re-run this cell to see whether training helped.

In [ ]:
def actual_topics(i):
    return {c["topic"] for c in analysed[i]["actual"]}


def hit_rates(members, ranker):
    if not members:
        return {"n": 0, "hit@1": None, "hit@3": None}
    ranked = [(ranker(i), actual_topics(i)) for i in members]
    return {"n": len(members),
            "hit@1": round(float(np.mean([bool(set(r[:1]) & a) for r, a in ranked])), 3),
            "hit@3": round(float(np.mean([bool(set(r[:3]) & a) for r, a in ranked])), 3)}


common = [t for t, _ in Counter(t for i in split["train"] for t in actual_topics(i)).most_common()]
n_followed = float(np.mean([len(actual_topics(i)) for i in kept]))
print(f"On average {n_followed:.1f} of the {len(TOPIC_ENUM)} topics followed each event.")
rank_by_power = lambda p: (lambda i: [t["topic"] for t in vote(analysed[i]["res"]["analogues"], p)])
powers = [1.0, 4.0, 8.0]
val_scores = {p: hit_rates(split["val"], rank_by_power(p)) for p in powers}
best_power = max(powers, key=lambda p: ((val_scores[p]["hit@1"] or 0), (val_scores[p]["hit@3"] or 0), -p))
print("Validation, relevance-weighted counting by weight power:", {p: (s["hit@1"], s["hit@3"]) for p, s in val_scores.items()})
print(f"Chosen power: {best_power} (set REL_POWER = {best_power} in step 1 to use it everywhere)\n")

report = [{"method": "most common topics", **hit_rates(split["test"], lambda i: common)},
          {"method": "counting, unweighted", **hit_rates(split["test"], rank_by_power(0.0))},
          {"method": f"counting, relevance-weighted (power {best_power})", **hit_rates(split["test"], rank_by_power(best_power))}]

if N_EVAL_LLM:
    llm_members = split["test"][:N_EVAL_LLM]
    run_parallel(lambda i: reason(i, analysed[i]["res"], analysed[i]["table"]), llm_members, "LLM answers")
    answers = {i: read_json(CACHE / "reasoning" / f"{_slug(ids[i])}.json") for i in llm_members}
    answered = [i for i in llm_members if answers[i]]
    row = hit_rates(answered, lambda i: [x["topic"] for x in answers[i]["likely_next"]])
    report.append({"method": f"LLM with retrieval ({'dry-run' if DRY_RUN else LLM_MODEL})", **row})
    if answered:
        report.append({"method": "counting, relevance-weighted — same events as the LLM row", **hit_rates(answered, rank_by_power(best_power))})

report_df = pd.DataFrame(report)
show(report_df)
report_df.to_csv(WORK / "evaluation.csv", index=False)
print(f"Saved to {WORK / 'evaluation.csv'}. With fewer than about 200 test events, small differences between rows are noise.")
if (report[0]["hit@1"] or 0) >= 0.8:
    print("Warning: always answering the most common topic is right "
          f"{report[0]['hit@1']:.0%} of the time, so this target cannot separate good methods from bad ones. "
          "Raise WINDOW_SIM_PCT, shorten DAYS_AFTER, or use a finer TOPICS list, then re-run from step 6.")
if DRY_RUN:
    print("DRY RUN: these numbers only show that the evaluation runs. They say nothing about the method.")

## 13. Fine-tune with LoRA

**This step was only smoke-tested** on a tiny random model on CPU (no GPU where the notebook was written).
The code path runs, but it has never trained a real Qwen, so expect to adjust memory settings.

What you need on the supercomputer:
- The model's original weights in Hugging Face format. Ollama's own model file cannot be trained;
  download the matching Qwen folder once on a node with internet and set `BASE_MODEL_HF` to it.
- `pip install transformers peft accelerate` in your environment.
- One GPU with enough memory for the model in 16-bit (about 2 GB per billion parameters, plus working room).

Set `RUN_FINETUNE = True` in step 1 to run it. Use real (not dry-run) training files.

In [ ]:
if RUN_FINETUNE:
    if DRY_RUN:
        raise RuntimeError("The training files were made in DRY_RUN mode. Re-run steps 10–11 with the LLM first.")
    import torch
    from peft import LoraConfig, get_peft_model
    from transformers import (AutoModelForCausalLM, AutoTokenizer, DataCollatorForLanguageModeling,
                              Trainer, TrainingArguments)

    tok = AutoTokenizer.from_pretrained(BASE_MODEL_HF)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    def encode_file(name):
        rows, too_long = [], 0
        for line in open(DATASET / f"{name}.jsonl", encoding="utf-8"):
            text = tok.apply_chat_template(json.loads(line)["messages"], tokenize=False, enable_thinking=False)
            enc = tok(text)
            if len(enc["input_ids"]) > MAX_TOKENS:
                too_long += 1          # cutting would remove the answer, so the example is skipped instead
                continue
            rows.append(enc)
        print(f"{name}: {len(rows)} examples, {too_long} skipped as longer than {MAX_TOKENS} tokens")
        return rows

    train_rows, val_rows = encode_file("train"), encode_file("val")
    on_gpu = torch.cuda.is_available()
    if not on_gpu:
        print("No GPU found. Training a real model on CPU is not practical; this only makes sense for a tiny test model.")
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_HF, torch_dtype=torch.bfloat16 if on_gpu else torch.float32,
                                                 device_map="auto" if on_gpu else None)
    model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, task_type="CAUSAL_LM",
                                             target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
    model.print_trainable_parameters()
    out_dir = WORK / "finetune"
    trainer = Trainer(
        model=model, train_dataset=train_rows, eval_dataset=val_rows or None,
        data_collator=DataCollatorForLanguageModeling(tok, mlm=False),
        args=TrainingArguments(output_dir=str(out_dir / "checkpoints"), num_train_epochs=TRAIN_EPOCHS,
                               learning_rate=LEARNING_RATE, per_device_train_batch_size=1, gradient_accumulation_steps=8,
                               logging_steps=10, save_strategy="epoch", bf16=on_gpu, report_to=[]))
    trainer.train()
    if val_rows:
        print("Validation loss:", trainer.evaluate()["eval_loss"])
    merged = model.merge_and_unload()
    merged.save_pretrained(out_dir / "merged")
    tok.save_pretrained(out_dir / "merged")
    print(f"Merged model saved in {out_dir / 'merged'}")
else:
    print("Skipped. Set RUN_FINETUNE = True in step 1 when the training files are ready.")

## 14. Pack the fine-tuned model for Ollama

Run these in a terminal on the supercomputer (also not run by the notebook's author). They need
[llama.cpp](https://github.com/ggerganov/llama.cpp) checked out for its conversion script.

```bash
# 1. Convert the merged model to the file format Ollama reads
python llama.cpp/convert_hf_to_gguf.py chain_runs/run_v1/finetune/merged \
       --outfile chain_runs/run_v1/finetune/event-chain.gguf --outtype q8_0

# 2. Write the Modelfile (the next cell does this) and register the model
ollama create event-chain -f chain_runs/run_v1/finetune/Modelfile

# 3. Check it answers
ollama run event-chain "Reply with the word ready."
```

Then set `LLM_MODEL = "event-chain"` in step 1 and re-run step 12. Keep the fine-tuned model only if its
row beats both the counting row and the untrained LLM row on the test block.

The deployed service is two parts: this model in Ollama, and a small API in front of it that runs steps 4–8
(search, windows, relevance, counting) and sends the step 9 prompt. The model on its own does not hold the
event history.

In [ ]:
modelfile = WORK / "finetune" / "Modelfile"
modelfile.parent.mkdir(parents=True, exist_ok=True)
modelfile.write_text(
    "FROM ./event-chain.gguf\n"
    "PARAMETER temperature 0\n"
    f"PARAMETER num_ctx {LLM_NUM_CTX}\n"
    f'SYSTEM """{SYS_REASON}"""\n', encoding="utf-8")
print(f"Modelfile written to {modelfile}")
print("\nFiles produced by this run:")
for p in [WORK / "base_events", DATASET / "train.jsonl", DATASET / "val.jsonl", DATASET / "test.jsonl",
          DATASET / "split_manifest.json", WORK / "evaluation.csv", WORK / "failures.jsonl", CACHE]:
    print(f"  {'ok ' if p.exists() else '-- '} {p}")